In [ ]:
import csv
import json
import os
import re
from collections import Counter
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = "https://www.thebluealliance.com/api/v3"

# Inclusive year range to process.
START_YEAR = 2023
END_YEAR = 2026

# TBA event_type 99 is Offseason.
OFFSEASON_EVENT_TYPE = 99


def load_tba_key(env_path: str = ".env") -> str:
    key = os.getenv("TBA_KEY")
    if key:
        return key.strip()

    p = Path(env_path)
    if not p.exists():
        raise FileNotFoundError(f"Could not find {env_path} and TBA_KEY is not set in environment.")

    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("TBA_KEY="):
            value = line.split("=", 1)[1].strip().strip('"').strip("'")
            if value:
                return value

    raise ValueError("TBA_KEY was not found in .env")


def tba_get(path: str, key: str):
    url = f"{BASE_URL}{path}"
    headers = {
        "X-TBA-Auth-Key": key,
        "User-Agent": "cs2621-term-project/1.0 (+python)",
        "Accept": "application/json",
    }
    req = Request(url, headers=headers)
    try:
        with urlopen(req, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP error {e.code} for {url}: {body}") from e
    except URLError as e:
        raise RuntimeError(f"Network error for {url}: {e}") from e


def is_offseason_event(event_obj):
    event_type = event_obj.get("event_type")
    event_type_string = (event_obj.get("event_type_string") or "").strip().lower()
    return event_type == OFFSEASON_EVENT_TYPE or event_type_string == "offseason"


def parse_alliance_number(alliance_obj):
    number = alliance_obj.get("number")
    if isinstance(number, int):
        return number

    name = alliance_obj.get("name") or ""
    match = re.search(r"(\d+)", name)
    if match:
        return int(match.group(1))

    return None


def team_num(team_key):
    if isinstance(team_key, str):
        return team_key.replace("frc", "").strip()
    return team_key


def build_alliance_slots(alliances):
    slot_rows = []
    used_slots = set()

    for idx, alliance in enumerate(alliances, start=1):
        slot = parse_alliance_number(alliance)
        if slot is None or slot in used_slots:
            slot = idx
            while slot in used_slots:
                slot += 1
        used_slots.add(slot)

        picks = alliance.get("picks") or []
        slot_rows.append(
            {
                "slot": slot,
                "name": alliance.get("name"),
                "status": (alliance.get("status") or {}).get("status"),
                "picks": picks,
                "pick_set": set(picks),
            }
        )

    return slot_rows


def infer_winner_slot_from_matches(event_key, slot_rows, tba_key):
    matches = tba_get(f"/event/{event_key}/matches", tba_key)
    if not matches:
        return None

    playoff_matches = [m for m in matches if m.get("comp_level") in {"f", "sf", "qf", "ef"}]
    if not playoff_matches:
        return None

    # Prefer finals first; if unavailable, fall back to deepest available playoff level.
    for level in ["f", "sf", "qf", "ef"]:
        level_matches = [m for m in playoff_matches if m.get("comp_level") == level]
        if level_matches:
            playoff_matches = level_matches
            break

    wins_by_slot = Counter()
    for match in playoff_matches:
        winner_side = match.get("winning_alliance")
        if winner_side not in {"red", "blue"}:
            continue

        side_info = (match.get("alliances") or {}).get(winner_side) or {}
        winner_team_keys = set(side_info.get("team_keys") or [])
        if not winner_team_keys:
            continue

        best_slot = None
        best_overlap = -1
        for slot_row in slot_rows:
            overlap = len(winner_team_keys & slot_row["pick_set"])
            if overlap > best_overlap:
                best_overlap = overlap
                best_slot = slot_row["slot"]

        if best_slot is not None and best_overlap > 0:
            wins_by_slot[best_slot] += 1

    if not wins_by_slot:
        return None

    return wins_by_slot.most_common(1)[0][0]


def get_winning_alliance_number(event_key, slot_rows, tba_key):
    for slot_row in slot_rows:
        if slot_row["status"] == "won":
            return slot_row["slot"]

    return infer_winner_slot_from_matches(event_key, slot_rows, tba_key)


def normalize_event_row(event_key, year, alliances, tba_key):
    slot_rows = build_alliance_slots(alliances)

    row = {
        "year": year,
        "event_key": event_key,
        "winning_alliance": get_winning_alliance_number(event_key, slot_rows, tba_key),
    }

    for slot_row in sorted(slot_rows, key=lambda r: r["slot"]):
        picks = slot_row["picks"]
        slot = slot_row["slot"]
        row[f"alliance_{slot}_captain"] = team_num(picks[0]) if len(picks) > 0 else None
        row[f"alliance_{slot}_pick_2"] = team_num(picks[1]) if len(picks) > 1 else None
        row[f"alliance_{slot}_pick_3"] = team_num(picks[2]) if len(picks) > 2 else None

    return row


def fetch_event_rows_for_year(year: int, tba_key: str):
    events = tba_get(f"/events/{year}", tba_key)
    rows = []

    for event in events:
        if is_offseason_event(event):
            continue

        event_key = event.get("key")
        if not event_key:
            continue

        alliances = tba_get(f"/event/{event_key}/alliances", tba_key)
        if not alliances:
            continue

        rows.append(normalize_event_row(event_key=event_key, year=year, alliances=alliances, tba_key=tba_key))

    return rows


def fetch_event_rows_for_years(start_year: int, end_year: int, tba_key: str):
    all_rows = []

    for year in range(start_year, end_year + 1):
        year_rows = fetch_event_rows_for_year(year, tba_key)
        all_rows.extend(year_rows)
        print(f"{year}: {len(year_rows)} event rows")

    return all_rows


def build_fieldnames(rows):
    fieldnames = ["year", "event_key", "winning_alliance"]

    alliance_nums = set()
    for row in rows:
        for key in row.keys():
            match = re.match(r"alliance_(\d+)_(captain|pick_2|pick_3)$", key)
            if match:
                alliance_nums.add(int(match.group(1)))

    for n in sorted(alliance_nums):
        fieldnames.append(f"alliance_{n}_captain")
        fieldnames.append(f"alliance_{n}_pick_2")
        fieldnames.append(f"alliance_{n}_pick_3")

    return fieldnames


def write_csv(rows, output_path="playoff_alliances_with_winners.csv"):
    fieldnames = build_fieldnames(rows)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def main():
    tba_key = load_tba_key(".env")
    rows = fetch_event_rows_for_years(START_YEAR, END_YEAR, tba_key)
    write_csv(rows)

    missing_winners = sum(1 for row in rows if row.get("winning_alliance") is None)
    print(f"Wrote {len(rows)} event rows to playoff_alliances_with_winners.csv")
    print(f"Events with missing winner after fallback: {missing_winners}")
    if rows:
        print("Sample row:")
        print(rows[0])


main()

2022: 184 event rows
2023: 185 event rows
2024: 191 event rows
2025: 204 event rows
2026: 202 event rows
Wrote 966 event rows to playoff_alliances_with_winners.csv
Events with missing winner after fallback: 0
Sample row:
{'year': 2022, 'event_key': '2022alhu', 'winning_alliance': 1, 'alliance_1_captain': '364', 'alliance_1_pick_2': '59', 'alliance_1_pick_3': '3959', 'alliance_2_captain': '6517', 'alliance_2_pick_2': '1710', 'alliance_2_pick_3': '6652', 'alliance_3_captain': '325', 'alliance_3_pick_2': '5484', 'alliance_3_pick_3': '6936', 'alliance_4_captain': '6823', 'alliance_4_pick_2': '144', 'alliance_4_pick_3': '1038', 'alliance_5_captain': '7111', 'alliance_5_pick_2': '1102', 'alliance_5_pick_3': '2973', 'alliance_6_captain': '3653', 'alliance_6_pick_2': '108', 'alliance_6_pick_3': '3814', 'alliance_7_captain': '3201', 'alliance_7_pick_2': '6366', 'alliance_7_pick_3': '120', 'alliance_8_captain': '5410', 'alliance_8_pick_2': '801', 'alliance_8_pick_3': '2556'}
